# Mujassam — Base Model vs Fine-Tuned Model

This notebook creates the **before vs after fine-tuning evidence** for the final project.

It compares:

- **Baseline:** `Qwen2.5-Coder-7B-Instruct` with the QLoRA adapter disabled.
- **Mujassam:** the same base model with the final trained QLoRA adapter enabled.

Both conditions use the same:
- prompts
- RAG
- system prompt
- CadQuery executor
- validator
- feature gate
- greedy decoding
- one generation attempt

So the only model difference is **before QLoRA vs after QLoRA**.

## Test sets
- `in_family_20` — final model previously achieved **20/20**
- `in_family_f01_f12` — final model previously achieved **12/12**

### Upload when asked
1. `Mujassam_QWEN7B_V5_project.zip`
2. `mujassam_v5_A_7B_r16_adapter.zip`

Run all cells from top to bottom.

In [1]:
TEST_KEYS = ["in_family_20", "in_family_f01_f12"]

FIRST_TRY_ONLY = True
ROUNDS = 1 if FIRST_TRY_ONLY else 4

print("Tests:", TEST_KEYS)
print("Attempts per case:", ROUNDS)

Tests: ['in_family_20', 'in_family_f01_f12']
Attempts per case: 1


In [2]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || true

import torch
assert torch.cuda.is_available(), "Choose a GPU runtime in Colab."

GPU_NAME = torch.cuda.get_device_name(0)
print("GPU:", GPU_NAME)

NVIDIA A100-SXM4-40GB, 40960 MiB
GPU: NVIDIA A100-SXM4-40GB


In [3]:
import os, sys, zipfile, shutil, json, hashlib, glob, math
from pathlib import Path
from google.colab import files

PROJECT = Path("/content/mujassam_mvp_project")
if PROJECT.exists():
    shutil.rmtree(PROJECT)

print("Upload Mujassam_QWEN7B_V5_project.zip")
uploaded = files.upload()
project_zip = next(iter(uploaded))

with zipfile.ZipFile(project_zip) as z:
    z.extractall("/content")

assert PROJECT.exists(), f"Expected project folder at {PROJECT}"

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
print("Working directory:", os.getcwd())

DRIVE = None
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive/mujassam_baseline_compare")
    DRIVE.mkdir(parents=True, exist_ok=True)
    print("Drive backup folder:", DRIVE)
except Exception as exc:
    print("Drive not mounted; results will still download.")
    print(exc)

Upload Mujassam_QWEN7B_V5_project.zip


Saving Mujassam_QWEN7B_V5_project.zip to Mujassam_QWEN7B_V5_project.zip
Working directory: /content/mujassam_mvp_project
Mounted at /content/drive
Drive backup folder: /content/drive/MyDrive/mujassam_baseline_compare


In [4]:
!apt-get -qq install -y libgl1 libxrender1 libglu1-mesa > /dev/null 2>&1
!pip install -q "cadquery==2.8.0" "trimesh>=4.0" 2>&1 | tail -1
!pip install -q "transformers>=4.46" "accelerate>=1.0" "bitsandbytes>=0.44" "peft>=0.13" "trl>=0.12" "datasets>=3.0" "sentence-transformers>=3.0" "chromadb>=0.5" "gradio==6.29.0" 2>&1 | tail -1
!pip uninstall -y -q torchao || true

!python scripts/smoke_test.py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 5.2 MB/s eta 0:00:00
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.1 which is incompatible.
Smoke test passed. Gates hit: ['code', 'intent', 'pass']. STL: /content/mujassam_mvp_project/out/smoke/r3.stl
Scale guard passed: 100.0x -> /content/mujassam_mvp_project/out/scale_smoke/r1_scaled.step
Single-solid gate passed: ['geometry', 'pass'], faces=8


In [5]:
from mujassam.runtime_v3 import load_model, adapter_base

print("Upload mujassam_v5_A_7B_r16_adapter.zip")
uploaded = files.upload()
adapter_zip = next(iter(uploaded))

ADAPTER_ROOT = Path("/content/mujassam_final_adapter")
shutil.rmtree(ADAPTER_ROOT, ignore_errors=True)
ADAPTER_ROOT.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(adapter_zip) as z:
    z.extractall(ADAPTER_ROOT)

found = list(ADAPTER_ROOT.rglob("adapter_config.json"))
assert found, "adapter_config.json was not found in the uploaded adapter ZIP."

ADAPTER_DIR = str(found[0].parent)
BASE_MODEL = adapter_base(ADAPTER_DIR)

model, tok = load_model(ADAPTER_DIR)

print("Base model:", BASE_MODEL)
print("Adapter:", ADAPTER_DIR)
print("Loaded successfully.")

Upload mujassam_v5_A_7B_r16_adapter.zip


Saving mujassam_v5_A_7B_r16_adapter.zip to mujassam_v5_A_7B_r16_adapter.zip


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Base model: Qwen/Qwen2.5-Coder-7B-Instruct
Adapter: /content/mujassam_final_adapter
Loaded successfully.


In [6]:
from mujassam.runtime_v3 import build_final_pipeline
from mujassam.evaluate import load_prompts, run_generative_eval, report_view
from mujassam.monitor import RunLog

Path("out").mkdir(exist_ok=True)

def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

LOCKED = json.loads(Path("data/v5/locked_tests.json").read_text())
SETS = {}

for key in TEST_KEYS:
    info = LOCKED[key]
    assert sha(info["path"]) == info["sha256"], f"{info['path']} changed since it was locked."
    rows = load_prompts(Path(info["path"]))
    if "ids" in info:
        rows = [r for r in rows if r["id"] in info["ids"]]
    assert len(rows) == info["n"], (key, len(rows), info["n"])
    SETS[key] = rows

print("Locked sets:", {k: len(v) for k, v in SETS.items()})

run_log = RunLog(Path("out/baseline_compare_runs.jsonl"), alert_after=10**6)
RESULTS = {}

def evaluate(label, key, pipeline, adapter_off=False):
    tag = f"{label}_{key}"

    if adapter_off:
        with model.disable_adapter():
            run = run_generative_eval(
                pipeline, SETS[key], tag, run_log=run_log
            )
    else:
        run = run_generative_eval(
            pipeline, SETS[key], tag, run_log=run_log
        )

    size_records = report_view(run["records"])

    n = len(run["records"])
    size_pass = sum(int(r["ok"]) for r in size_records)
    gate_pass = sum(int(r["ok"]) for r in run["records"])

    RESULTS[(label, key)] = {
        "n": n,
        "size_pass": size_pass,
        "gate_pass": gate_pass,
        "records": run["records"],
        "size_records": size_records,
    }

    print(
        f"{label:12s} | {key:22s} | "
        f"size {size_pass}/{n} ({size_pass/n:.1%}) | "
        f"size+holes {gate_pass}/{n} ({gate_pass/n:.1%})"
    )

    return run

Locked sets: {'in_family_20': 20, 'in_family_f01_f12': 12}


In [9]:
# Build ONE shared pipeline.
# Everything stays identical between baseline and final model:
# RAG, prompt, executor, validator, feature gate, decoding.
# Only the QLoRA adapter is switched OFF / ON.

PIPE = build_final_pipeline(
    model,
    tok,
    validator="v3",
    feature_check="gate",
    n_candidates=1,
    max_rounds=ROUNDS,
    model_id="Mujassam baseline-vs-final comparison",
)

print("Shared pipeline ready.")
print("Baseline = QLoRA OFF")
print("Final    = QLoRA ON")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Shared pipeline ready.
Baseline = QLoRA OFF
Final    = QLoRA ON


In [11]:
for key in TEST_KEYS:
    print("\n" + "=" * 90)
    print("TEST SET:", key)
    print("=" * 90)

    print("\n--- BASELINE: QLoRA OFF ---")
    evaluate("BASE", key, PIPE, adapter_off=True)

    print("\n--- MUJASSAM: QLoRA ON ---")
    evaluate("FINAL", key, PIPE, adapter_off=False)


TEST SET: in_family_20

--- BASELINE: QLoRA OFF ---
n01  | FAIL | rounds=1 | programs=1 | bbox=None
n02  | PASS | rounds=1 | programs=1 | bbox=[35.0, 35.0, 7.5]
n03  | FAIL | rounds=1 | programs=1 | bbox=[31.0, 31.0, 12.0]
n04  | FAIL | rounds=1 | programs=1 | bbox=None
n05  | FAIL | rounds=1 | programs=1 | bbox=None
n06  | FAIL | rounds=1 | programs=1 | bbox=[34.0, 34.0, 34.0]
n07  | FAIL | rounds=1 | programs=1 | bbox=[98.0, 23.0, 13.0]
n08  | FAIL | rounds=1 | programs=1 | bbox=None
n09  | FAIL | rounds=1 | programs=1 | bbox=[55.0, 37.0, 39.0]
n10  | FAIL | rounds=1 | programs=1 | bbox=None
n11  | FAIL | rounds=1 | programs=1 | bbox=None
n12  | FAIL | rounds=1 | programs=1 | bbox=None
n13  | FAIL | rounds=1 | programs=1 | bbox=None
n14  | FAIL | rounds=1 | programs=1 | bbox=None
n15  | FAIL | rounds=1 | programs=1 | bbox=None
n16  | FAIL | rounds=1 | programs=1 | bbox=None
n17  | FAIL | rounds=1 | programs=1 | bbox=[37.0, 31.0, 3.0]
n18  | FAIL | rounds=1 | programs=1 | bbox=[55.0,

In [12]:
import pandas as pd

rows = []

for key in TEST_KEYS:
    base = RESULTS[("BASE", key)]
    final = RESULTS[("FINAL", key)]
    n = base["n"]
    assert final["n"] == n

    rows.append({
        "Evaluation set": key,
        "Cases": n,
        "Base size": f'{base["size_pass"]}/{n}',
        "Base size %": round(100 * base["size_pass"] / n, 1),
        "Final size": f'{final["size_pass"]}/{n}',
        "Final size %": round(100 * final["size_pass"] / n, 1),
        "Size improvement (pp)": round(100 * (final["size_pass"] - base["size_pass"]) / n, 1),
        "Base size+holes": f'{base["gate_pass"]}/{n}',
        "Base size+holes %": round(100 * base["gate_pass"] / n, 1),
        "Final size+holes": f'{final["gate_pass"]}/{n}',
        "Final size+holes %": round(100 * final["gate_pass"] / n, 1),
        "Gate improvement (pp)": round(100 * (final["gate_pass"] - base["gate_pass"]) / n, 1),
    })

summary = pd.DataFrame(rows)
display(summary)

summary.to_csv("out/baseline_vs_final_summary.csv", index=False)
Path("out/baseline_vs_final_summary.json").write_text(json.dumps(rows, indent=2))

print("Saved summary files.")

,Evaluation set,Cases,Base size,Base size %,Final size,Final size %,Size improvement (pp),Base size+holes,Base size+holes %,Final size+holes,Final size+holes %,Gate improvement (pp)
0,in_family_20,20,7/20,35.0,20/20,100.0,65.0,2/20,10.0,20/20,100.0,90.0
1,in_family_f01_f12,12,1/12,8.3,12/12,100.0,91.7,0/12,0.0,12/12,100.0,100.0


Saved summary files.


In [13]:
print("\nREADME-READY BEFORE vs AFTER TABLE\n")
print("| Evaluation Set | Base Qwen2.5-Coder-7B | Mujassam Fine-Tuned |")
print("|---|---:|---:|")

pretty_names = {
    "in_family_20": "In-Domain Regression",
    "in_family_f01_f12": "Second In-Domain Regression",
}

for key in TEST_KEYS:
    b = RESULTS[("BASE", key)]
    f = RESULTS[("FINAL", key)]
    n = b["n"]
    pretty = pretty_names.get(key, key)

    print(
        f"| **{pretty}** | "
        f"**{b['size_pass']}/{n} · {b['size_pass']/n:.1%}** | "
        f"**{f['size_pass']}/{n} · {f['size_pass']/n:.1%}** |"
    )

bundle = Path("/content/mujassam_baseline_compare_results.zip")
keep = [
    "out/baseline_vs_final_summary.csv",
    "out/baseline_vs_final_summary.json",
    "out/baseline_compare_runs.jsonl",
]

with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for f in keep:
        if Path(f).exists():
            z.write(f)

if DRIVE is not None:
    shutil.copy2(bundle, DRIVE / bundle.name)
    shutil.copy2("out/baseline_vs_final_summary.csv", DRIVE / "baseline_vs_final_summary.csv")

print("\nDownloading:", bundle)
files.download(str(bundle))


README-READY BEFORE vs AFTER TABLE

| Evaluation Set | Base Qwen2.5-Coder-7B | Mujassam Fine-Tuned |
|---|---:|---:|
| **In-Domain Regression** | **7/20 · 35.0%** | **20/20 · 100.0%** |
| **Second In-Domain Regression** | **1/12 · 8.3%** | **12/12 · 100.0%** |

Downloading: /content/mujassam_baseline_compare_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>